In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from datasets import Dataset
from datasets import Value

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import matplotlib.pyplot as plt
import seaborn as sns
import re
import spacy
import joblib

from collections import Counter


c:\Users\swoye\anaconda3\envs\finance_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
transaction_df = pd.read_csv('final_transactions_data.csv')

le = LabelEncoder()
transaction_df['Category_Enc'] = le.fit_transform(transaction_df['Category'])

transaction_df['Date'] = pd.to_datetime(transaction_df['Date'], dayfirst=False)

transaction_df['Month'] = transaction_df['Date'].dt.month  
transaction_df['Day'] = transaction_df['Date'].dt.day  

In [3]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\swoye\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\swoye\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\swoye\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\swoye\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\swoye\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [4]:
from nltk.tokenize import word_tokenize
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [5]:
def textPreprocessing(text):
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.lower()  
    text = word_tokenize(text)
    return text

def stopwordRemoval(text):
    text = [word for word in text if word not in stop_words]
    return text

def lemmatization(text):
    text = [lemmatizer.lemmatize(word) for word in text]
    return text 

def posTagging(text):
    text = nltk.pos_tag(text)
    text = [word for word, tag in text if tag.startswith('N') or tag.startswith('V')]   
    return text

def cleanTextPipeline(text):
    text = textPreprocessing(text)
    text = stopwordRemoval(text)
    text = lemmatization(text)
    text = posTagging(text)
    text = ' '.join(text)
    return text

transaction_df['processedDescription'] = transaction_df["Description"].apply(lambda x : cleanTextPipeline(x))

MemoryError: 

In [ ]:
nlp = spacy.load('en_core_web_sm')

In [ ]:
docs = [nlp(desc) for desc in transaction_df['Description']]

In [ ]:
keywords_type = []

for doc in docs:
    types = [ent.label_ for ent in doc.ents]
    keywords_type.append(types)

transaction_df['keywords_type'] = keywords_type
transaction_df['keywords_type_str'] = transaction_df['keywords_type'].apply(lambda x: ', '.join(x) if x else '')

In [ ]:
transaction_df['combinedText'] = (
    transaction_df['processedDescription'].fillna('') + ' ' +
    transaction_df['keywords_types_str'].fillna('') + ' '
)

KeyError: 'processedDescription'

In [ ]:
print(transaction_df['combinedText'].head())

0        ordered foodmandu NORP 3047.69 0.0 
1     order placed daraz PERSON 2630.51 0.0 
2       service tranquil spa FAC 4055.8 0.0 
3    purchased grocery mart ORG 2430.92 0.0 
4        payment tuition class  1800.66 0.0 
Name: combinedText, dtype: object


In [ ]:
transaction_df = transaction_df[['combinedText', 'Category_Enc']]

In [ ]:
all_texts = transaction_df['combinedText'].astype(str)
all_labels = transaction_df['Category_Enc'].astype(int)

In [ ]:
dataset = Dataset.from_dict({
    "text": list(all_texts),
    "label": list(all_labels)
})
split_dataset = dataset.shuffle(seed=42).train_test_split(test_size=0.2)
train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

split_train = train_dataset.train_test_split(test_size=0.1)
train_dataset = split_train["train"]
val_dataset = split_train["test"]

In [ ]:
model_path = "sentence-transformers/all-MiniLM-l6-v2"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_path)

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=64
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

columns = ["input_ids", "attention_mask", "label"]
train_dataset.set_format(type="torch", columns=columns)
val_dataset.set_format(type="torch", columns=columns)
test_dataset.set_format(type="torch", columns=columns)

Map:   0%|          | 0/7199 [00:00<?, ? examples/s]

Map: 100%|██████████| 2000/2000 [00:00<00:00, 11738.75 examples/s]


In [ ]:
num_labels = len(le.classes_)
model = AutoModelForSequenceClassific
ation.from_pretrained(
    model_path, num_labels=num_labels
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at sentence-transformers/all-MiniLM-l6-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
device = torch.device("cpu")
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 384, padding_idx=0)
      (position_embeddings): Embedding(512, 384)
      (token_type_embeddings): Embedding(2, 384)
      (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-5): 6 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=384, out_features=384, bias=True)
              (key): Linear(in_features=384, out_features=384, bias=True)
              (value): Linear(in_features=384, out_features=384, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=384, out_features=384, bias=True)
              (LayerNorm): LayerNorm((384,), eps=1e-1

In [ ]:
def compute_metrics(p):
    preds = torch.argmax(torch.tensor(p.predictions), axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        p.label_ids, preds, average="weighted"
    )
    acc = accuracy_score(p.label_ids, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

In [ ]:
for param in model.base_model.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.05,
    logging_dir="./logs",
    disable_tqdm=False,
    load_best_model_at_end=True,
    no_cuda=True,
    dataloader_num_workers=0,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)
trainer.train()

c:\Users\swoye\anaconda3\envs\finance_env\Lib\site-packages\transformers\training_args.py:1636: FutureWarning: using `no_cuda` is deprecated and will be removed in version 5.0 of 🤗 Transformers. Use `use_cpu` instead
  warnings.warn(
C:\Users\swoye\AppData\Local\Temp\ipykernel_12468\2292180430.py:16: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,2.384600,2.351392,0.613750,0.600196,0.757614,0.613750


KeyboardInterrupt: 

In [ ]:
model.save_pretrained("minilm_transaction_cpu_model")
tokenizer.save_pretrained("minilm_transaction_cpu_model")
joblib.dump(le, "label_encoder.pkl")

print("✅ Training complete! Model and label encoder saved.")

✅ Training complete! Model and label encoder saved.


In [ ]:
print(Counter(train_dataset['label']))
print(Counter(val_dataset['label']))

Counter({tensor(2): 1, tensor(6): 1, tensor(3): 1, tensor(1): 1, tensor(2): 1, tensor(1): 1, tensor(2): 1, tensor(0): 1, tensor(10): 1, tensor(9): 1, tensor(8): 1, tensor(8): 1, tensor(5): 1, tensor(7): 1, tensor(2): 1, tensor(8): 1, tensor(8): 1, tensor(0): 1, tensor(8): 1, tensor(6): 1, tensor(6): 1, tensor(0): 1, tensor(4): 1, tensor(7): 1, tensor(2): 1, tensor(4): 1, tensor(8): 1, tensor(8): 1, tensor(4): 1, tensor(3): 1, tensor(3): 1, tensor(9): 1, tensor(7): 1, tensor(8): 1, tensor(7): 1, tensor(0): 1, tensor(8): 1, tensor(1): 1, tensor(3): 1, tensor(2): 1, tensor(7): 1, tensor(6): 1, tensor(2): 1, tensor(2): 1, tensor(7): 1, tensor(9): 1, tensor(1): 1, tensor(6): 1, tensor(4): 1, tensor(0): 1, tensor(2): 1, tensor(5): 1, tensor(0): 1, tensor(4): 1, tensor(2): 1, tensor(2): 1, tensor(3): 1, tensor(0): 1, tensor(0): 1, tensor(7): 1, tensor(0): 1, tensor(8): 1, tensor(0): 1, tensor(8): 1, tensor(3): 1, tensor(10): 1, tensor(6): 1, tensor(10): 1, tensor(6): 1, tensor(2): 1, tensor(1